# 04 — Household Segmentation

This notebook loads the rule-based household segments produced by
`src.segmentation.assign_segments()`.  Segments are computed on RFM + promotional
behavior features derived from beverage activity only, using a priority hierarchy
that avoids double-counting high-spend households into behavioral tags.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
print('Ready')

In [ ]:
from src.data_cleaning import PROCESSED_DIR

seg = pd.read_parquet(PROCESSED_DIR / 'household_segments.parquet')
print(f'Total households in segment table: {len(seg):,}')
print(f'Columns: {list(seg.columns)}')
print()
print('Segment counts:')
print(seg['segment'].value_counts().to_string())

In [ ]:
# Segment KPI summary
seg_order = ['High-Value Beverage Buyers', 'Promo-Sensitive Shoppers',
             'Coupon-Driven Shoppers', 'Low-Engagement Shoppers']

total_bev_rev = seg['beverage_spend'].sum()

summary_rows = []
for s in seg_order:
    grp = seg[seg['segment'] == s]
    summary_rows.append({
        'segment': s,
        'n_households': len(grp),
        'pct_households': round(len(grp) / len(seg) * 100, 1),
        'bev_revenue': round(grp['beverage_spend'].sum(), 0),
        'pct_bev_revenue': round(grp['beverage_spend'].sum() / total_bev_rev * 100, 1),
        'avg_bev_spend': round(grp['beverage_spend'].mean(), 2),
        'avg_promo_share': round(grp['promo_purchase_share'].mean() * 100, 1),
        'avg_coupon_rate': round(grp['coupon_usage_rate'].mean() * 100, 2),
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

In [ ]:
# Feature distributions by segment
feature_cols = ['beverage_spend', 'promo_purchase_share', 'coupon_usage_rate',
                'repeat_purchase_rate', 'avg_basket_value']
seg_means = seg.groupby('segment')[feature_cols].mean().round(3)
print('Mean feature values by segment:')
print(seg_means.to_string())

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pathlib

CHARTS = pathlib.Path('..') / 'outputs' / 'charts'

seg_order = ['High-Value Beverage Buyers', 'Promo-Sensitive Shoppers',
             'Coupon-Driven Shoppers', 'Low-Engagement Shoppers']
seg_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#9467bd']

counts = seg['segment'].value_counts().reindex(seg_order)
rev_share = (seg.groupby('segment')['beverage_spend'].sum() / seg['beverage_spend'].sum() * 100).reindex(seg_order).round(1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

bars_a = axes[0].bar(seg_order, counts.values, color=seg_colors, edgecolor='white')
for bar, val in zip(bars_a, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
                 str(val), ha='center', va='bottom', fontsize=9)
axes[0].set_title('Household Count by Segment', fontsize=11)
axes[0].set_ylabel('Households', fontsize=10)
axes[0].set_xticks(range(len(seg_order)))
axes[0].set_xticklabels(seg_order, rotation=15, ha='right', fontsize=8.5)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

bars_b = axes[1].bar(seg_order, rev_share.values, color=seg_colors, edgecolor='white')
for bar, val in zip(bars_b, rev_share.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val}%', ha='center', va='bottom', fontsize=9)
axes[1].set_title('Beverage Revenue Share by Segment', fontsize=11)
axes[1].set_ylabel('Revenue Share (%)', fontsize=10)
axes[1].set_xticks(range(len(seg_order)))
axes[1].set_xticklabels(seg_order, rotation=15, ha='right', fontsize=8.5)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

fig.suptitle('Household Segmentation — 2,487 Beverage-Buying Households', fontsize=11, y=1.01)
plt.tight_layout()
fig.savefig(CHARTS / 'segments_count_revenue.png', dpi=130, bbox_inches='tight')
plt.close(fig)
print('Saved segments_count_revenue.png')

## Summary — Household Segmentation

- **2,487 households** purchased beverages (99.5% of all 2,500 panel households).
- Four segments using rule-based RFM + promo behavior (priority order prevents double-counting):

| Segment | HHs | % HHs | % Bev Revenue | Avg Spend |
|---|---|---|---|---|
| High-Value Beverage Buyers | 622 | 25.0% | 63.6% | $676.51 |
| Promo-Sensitive Shoppers | 1,554 | 62.5% | 31.6% | $134.55 |
| Coupon-Driven Shoppers | 123 | 4.9% | 2.3% | $123.11 |
| Low-Engagement Shoppers | 188 | 7.6% | 2.5% | $86.75 |

- The **High-Value segment** (25% of households) drives **64% of beverage revenue** — the primary
  retention target for loyalty investment.
- **Promo-Sensitive shoppers** (62.5% of households) account for 32% of revenue — they respond
  to discounts but their high promo dependency suppresses net margin.
- **Coupon redemption rate is only 1.86%** — coupons are a minor tool relative to shelf discounts.